In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" to "Questionnaire"
# if that is the folder name on your machine.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_I.xpt"
csv_file = DATA_DIR / "ALQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 ALQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ101",
    "ALQ110",
    "ALQ120Q",
    "ALQ120U",
    "ALQ130",
    "ALQ141Q",
    "ALQ141U",
    "ALQ151",
    "ALQ160"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ALQ_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 5735:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 5,735 rows x 10 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached ALQ_I:
#
#     Genuine decimal variables = NONE
#     String variables          = NONE
#
#
# All populated values are whole-number:
#
#     IDs
#     response codes
#     quantities
#     units
#
#
# Therefore all 10 fields can ultimately be nullable Int64.
#
#
# IMPORTANT:
#
# This applies ONLY to ALQ_I.
#
# It does NOT change the broader NHANES fidelity rule:
#
#     Dietary genuine decimals -> AS IS
#     Laboratory decimals      -> AS IS
#     Examination measurements -> AS IS
#     Survey/sample weights     -> AS IS
#
# Never globally round other NHANES datasets.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 10:

    raise ValueError(
        "Expected exactly 10 whole-number/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# NEVER globally replace this before validating
# the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 ALQ_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached ALQ_I.xpt:
#
#     ALQ120Q      845
#     ALQ141Q    2,025
#     ALQ160       969
#
# TOTAL         3,839
#
#
# These variables legitimately allow zero:
#
#
# ALQ120Q:
#     "ENTER '0' FOR NEVER"
#
# ALQ141Q:
#     "ENTER '0' FOR NONE"
#
# ALQ160:
#     "ENTER '0' FOR NEVER"
#
#
# Therefore this exact tiny-value pattern can safely
# be restored to real numeric zeros.
#

EXPECTED_TINY_COUNTS = {
    "ALQ120Q": 845,
    "ALQ141Q": 2025,
    "ALQ160": 969
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated ALQ_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 3839:

    raise ValueError(
        f"Unexpected tiny-value total: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 3:

    raise ValueError(
        "Expected tiny values in exactly 3 variables."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 3,839-value tiny-zero pattern "
    "across 3 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only the variables whose complete artifact
# pattern has already been validated.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 3839:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df is the exact corrected-XPT fidelity source.
#
# ONLY these changes are permitted:
#
#     validated tiny value -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is ONLY a diagnostic comparison.
#
# It does NOT modify source values.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimals found "
        "in ALQ_I.\n\n"
        "They will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "in ALQ_I."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 12. VALIDATE ALL FIELDS ARE WHOLE-NUMBER
# ============================================================

#
# Prove every populated value is whole-number before
# converting to Int64.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All 10 ALQ_I variables are "
    "whole-number/code fields."
)


# ============================================================
# 13. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 ALQ_I variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 14. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 15. CDC CHECK - ALQ101
# ============================================================

#
# Had at least 12 alcoholic drinks in any one year?
#
#     1 = Yes          3,477
#     2 = No           1,728
#     7 = Refused          0
#     9 = Don't know       3
#
# Missing = 527
#

check_frequency(
    "ALQ101",
    {
        1: 3477,
        2: 1728,
        9: 3
    },
    527
)


# ============================================================
# 16. CDC CHECK - ALQ110
# ============================================================

#
# Had at least 12 drinks in lifetime?
#
#     1 = Yes         747
#     2 = No          979
#     7 = Refused       1
#     9 = Don't know    4
#
# Missing = 4,004
#

check_frequency(
    "ALQ110",
    {
        1: 747,
        2: 979,
        7: 1,
        9: 4
    },
    4004
)


# ============================================================
# 17. ALQ101 -> ALQ110 ROUTING CHECK
# ============================================================

#
# CDC skip logic:
#
# ALQ101 = 1:
#     skip to ALQ120Q
#
# ALQ101 = 2 / 9:
#     ALQ110 may be asked
#
#
# Direct attachment:
#
#     ALQ101=1 -> ALQ110 missing exactly
#
#     ALQ101=2 or 9 -> ALQ110 populated exactly
#

alq101_yes = (
    df["ALQ101"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alq110_missing = (
    df["ALQ110"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


bad_yes_routing = int(
    np.sum(
        alq101_yes
        &
        ~alq110_missing
    )
)


alq101_followup = (
    df["ALQ101"]
    .isin([2, 9])
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alq110_present = (
    df["ALQ110"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


followup_mismatch = int(
    np.sum(
        alq101_followup
        !=
        alq110_present
    )
)


print(
    "\n--- ALQ101 / ALQ110 ROUTING ---"
)

print(
    "ALQ101=1 with ALQ110 unexpectedly populated:",
    bad_yes_routing
)

print(
    "Follow-up availability mismatches:",
    followup_mismatch
)


if bad_yes_routing != 0:

    raise ValueError(
        "ALQ101=1 routing to ALQ110 differs."
    )


if followup_mismatch != 0:

    raise ValueError(
        "ALQ110 availability does not match "
        "ALQ101 follow-up routing."
    )


print(
    "PASS: ALQ101/ALQ110 routing validated."
)


# ============================================================
# 18. CDC CHECK - ALQ120Q
# ============================================================

#
# Frequency drinking alcohol during past 12 months.
#
# CDC:
#
#     0-365 = 4,221
#     777   =     0
#     999   =     3
#
# Missing = 1,511
#
#
# Direct XPT:
#
#     zero = 845
#
# These 845 values were originally decoded as tiny values.
#

alq120q_range = int(
    df["ALQ120Q"]
    .between(
        0,
        365
    )
    .sum()
)


alq120q_999 = int(
    df["ALQ120Q"]
    .eq(999)
    .sum()
)


alq120q_777 = int(
    df["ALQ120Q"]
    .eq(777)
    .sum()
)


alq120q_zero = int(
    df["ALQ120Q"]
    .eq(0)
    .sum()
)


alq120q_missing = int(
    df["ALQ120Q"]
    .isna()
    .sum()
)


print(
    "\n--- ALQ120Q ---"
)

print(
    "0-365:",
    f"{alq120q_range:,}"
)

print(
    "Zero:",
    f"{alq120q_zero:,}"
)

print(
    "777:",
    alq120q_777
)

print(
    "999:",
    alq120q_999
)

print(
    "Missing:",
    f"{alq120q_missing:,}"
)


if alq120q_range != 4221:

    raise ValueError(
        "ALQ120Q 0-365 range count differs."
    )


if alq120q_zero != 845:

    raise ValueError(
        "ALQ120Q zero count differs."
    )


if alq120q_777 != 0:

    raise ValueError(
        "Unexpected ALQ120Q refused values."
    )


if alq120q_999 != 3:

    raise ValueError(
        "ALQ120Q don't-know count differs."
    )


if alq120q_missing != 1511:

    raise ValueError(
        "ALQ120Q missing count differs."
    )


print(
    "PASS: ALQ120Q matches CDC."
)


# ============================================================
# 19. CDC CHECK - ALQ120U
# ============================================================

#
# Unit:
#
#     1 = Week     1,308
#     2 = Month    1,003
#     3 = Year     1,065
#
# Missing = 2,359
#

check_frequency(
    "ALQ120U",
    {
        1: 1308,
        2: 1003,
        3: 1065
    },
    2359
)


# ============================================================
# 20. ALQ120Q / ALQ120U ROUTING CHECK
# ============================================================

#
# ALQ120U should exist when ALQ120Q is a positive
# numeric frequency from 1 through 365.
#
# It should not be populated when:
#
#     ALQ120Q = 0
#     ALQ120Q = 999
#     ALQ120Q missing
#

alq120_positive = (
    df["ALQ120Q"]
    .between(
        1,
        365
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alq120u_present = (
    df["ALQ120U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


alq120_routing_mismatch = int(
    np.sum(
        alq120_positive
        !=
        alq120u_present
    )
)


print(
    "\n--- ALQ120Q / ALQ120U ROUTING ---"
)

print(
    "Routing mismatches:",
    alq120_routing_mismatch
)


if alq120_routing_mismatch != 0:

    raise ValueError(
        "ALQ120U availability does not match "
        "positive ALQ120Q responses."
    )


print(
    "PASS: ALQ120Q/ALQ120U routing validated."
)


# ============================================================
# 21. CDC CHECK - ALQ130
# ============================================================

#
# Average drinks/day during drinking days.
#
# CDC:
#
#     1-14 = 3,355
#     15   =    20   (15 or more)
#     777  =     0
#     999  =     4
#
# Missing = 2,356
#

alq130_range = int(
    df["ALQ130"]
    .between(
        1,
        14
    )
    .sum()
)


alq130_15 = int(
    df["ALQ130"]
    .eq(15)
    .sum()
)


alq130_777 = int(
    df["ALQ130"]
    .eq(777)
    .sum()
)


alq130_999 = int(
    df["ALQ130"]
    .eq(999)
    .sum()
)


alq130_missing = int(
    df["ALQ130"]
    .isna()
    .sum()
)


print(
    "\n--- ALQ130 ---"
)

print(
    "1-14:",
    f"{alq130_range:,}"
)

print(
    "15 or more:",
    alq130_15
)

print(
    "777:",
    alq130_777
)

print(
    "999:",
    alq130_999
)

print(
    "Missing:",
    f"{alq130_missing:,}"
)


if alq130_range != 3355:

    raise ValueError(
        "ALQ130 1-14 count differs."
    )


if alq130_15 != 20:

    raise ValueError(
        "ALQ130 15-or-more count differs."
    )


if alq130_777 != 0:

    raise ValueError(
        "Unexpected ALQ130 refused value."
    )


if alq130_999 != 4:

    raise ValueError(
        "ALQ130 don't-know count differs."
    )


if alq130_missing != 2356:

    raise ValueError(
        "ALQ130 missing count differs."
    )


print(
    "PASS: ALQ130 matches CDC."
)


# ============================================================
# 22. CDC CHECK - ALQ141Q
# ============================================================

#
# Days with 4/5 or more drinks during past 12 months.
#
# CDC:
#
#     0-365 = 3,365
#     777   =     1
#     999   =    11
#
# Missing = 2,358
#
#
# Direct attachment:
#
#     zero = 2,025
#
# Those 2,025 zeros were initially represented by
# the tiny XPORT value.
#

alq141q_range = int(
    df["ALQ141Q"]
    .between(
        0,
        365
    )
    .sum()
)


alq141q_zero = int(
    df["ALQ141Q"]
    .eq(0)
    .sum()
)


alq141q_777 = int(
    df["ALQ141Q"]
    .eq(777)
    .sum()
)


alq141q_999 = int(
    df["ALQ141Q"]
    .eq(999)
    .sum()
)


alq141q_missing = int(
    df["ALQ141Q"]
    .isna()
    .sum()
)


print(
    "\n--- ALQ141Q ---"
)

print(
    "0-365:",
    f"{alq141q_range:,}"
)

print(
    "Zero:",
    f"{alq141q_zero:,}"
)

print(
    "777:",
    alq141q_777
)

print(
    "999:",
    alq141q_999
)

print(
    "Missing:",
    f"{alq141q_missing:,}"
)


if alq141q_range != 3365:

    raise ValueError(
        "ALQ141Q 0-365 count differs."
    )


if alq141q_zero != 2025:

    raise ValueError(
        "ALQ141Q zero count differs."
    )


if alq141q_777 != 1:

    raise ValueError(
        "ALQ141Q refused count differs."
    )


if alq141q_999 != 11:

    raise ValueError(
        "ALQ141Q don't-know count differs."
    )


if alq141q_missing != 2358:

    raise ValueError(
        "ALQ141Q missing count differs."
    )


print(
    "PASS: ALQ141Q matches CDC."
)


# ============================================================
# 23. CDC CHECK - ALQ141U
# ============================================================

#
# Unit:
#
#     1 = Week      351
#     2 = Month     286
#     3 = Year      703
#
# Missing = 4,395
#

check_frequency(
    "ALQ141U",
    {
        1: 351,
        2: 286,
        3: 703
    },
    4395
)


# ============================================================
# 24. ALQ141Q / ALQ141U ROUTING CHECK
# ============================================================

#
# ALQ141U exists exactly for positive numeric
# ALQ141Q quantities from 1-365.
#

alq141_positive = (
    df["ALQ141Q"]
    .between(
        1,
        365
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alq141u_present = (
    df["ALQ141U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


alq141_routing_mismatch = int(
    np.sum(
        alq141_positive
        !=
        alq141u_present
    )
)


print(
    "\n--- ALQ141Q / ALQ141U ROUTING ---"
)

print(
    "Routing mismatches:",
    alq141_routing_mismatch
)


if alq141_routing_mismatch != 0:

    raise ValueError(
        "ALQ141U availability does not match "
        "positive ALQ141Q responses."
    )


print(
    "PASS: ALQ141Q/ALQ141U routing validated."
)


# ============================================================
# 25. CDC CHECK - ALQ151
# ============================================================

#
# Ever drank 4/5 or more drinks almost every day?
#
#     1 = Yes          725
#     2 = No         3,495
#     7 = Refused        1
#     9 = Don't know     1
#
# Missing = 1,513
#

check_frequency(
    "ALQ151",
    {
        1: 725,
        2: 3495,
        7: 1,
        9: 1
    },
    1513
)


# ============================================================
# 26. CDC CHECK - ALQ160
# ============================================================

#
# Number of times 4/5+ drinks within about 2 hours
# during past 30 days.
#
# CDC:
#
#     0-18 = 1,331
#     20   =    15   (20 or more)
#     777  =     0
#     999  =     6
#
# Missing = 4,383
#
#
# Direct attachment:
#
#     zero = 969
#
# These are legitimate "never" responses.
#

alq160_range = int(
    df["ALQ160"]
    .between(
        0,
        18
    )
    .sum()
)


alq160_zero = int(
    df["ALQ160"]
    .eq(0)
    .sum()
)


alq160_20 = int(
    df["ALQ160"]
    .eq(20)
    .sum()
)


alq160_777 = int(
    df["ALQ160"]
    .eq(777)
    .sum()
)


alq160_999 = int(
    df["ALQ160"]
    .eq(999)
    .sum()
)


alq160_missing = int(
    df["ALQ160"]
    .isna()
    .sum()
)


print(
    "\n--- ALQ160 ---"
)

print(
    "0-18:",
    f"{alq160_range:,}"
)

print(
    "Zero:",
    f"{alq160_zero:,}"
)

print(
    "20 or more:",
    alq160_20
)

print(
    "777:",
    alq160_777
)

print(
    "999:",
    alq160_999
)

print(
    "Missing:",
    f"{alq160_missing:,}"
)


if alq160_range != 1331:

    raise ValueError(
        "ALQ160 0-18 count differs."
    )


if alq160_zero != 969:

    raise ValueError(
        "ALQ160 zero count differs."
    )


if alq160_20 != 15:

    raise ValueError(
        "ALQ160 20-or-more count differs."
    )


if alq160_777 != 0:

    raise ValueError(
        "Unexpected ALQ160 refused values."
    )


if alq160_999 != 6:

    raise ValueError(
        "ALQ160 don't-know count differs."
    )


if alq160_missing != 4383:

    raise ValueError(
        "ALQ160 missing count differs."
    )


print(
    "PASS: ALQ160 matches CDC."
)


# ============================================================
# 27. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Before repair:
#
#     ordinary zeros = 0
#
#
# After repair the ONLY zero values should be:
#
#     ALQ120Q =   845
#     ALQ141Q = 2,025
#     ALQ160  =   969
#
# TOTAL      = 3,839
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 3839:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 3,839 validated restorations."
)


# ============================================================
# 28. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY permitted numerical changes are:
#
#     validated tiny value -> 0
#
# Every other value must remain numerically identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 10 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# ALQ_I contains no genuine decimal variables.
#
# Still use the standard fidelity export:
#
#     NO df.round(...)
#     NO float_format=
#
#
# This preserves your consistent NHANES conversion method.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 30. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 10 ALQ_I variables."
)


# ============================================================
# 32. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 33. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 35. RAW CSV .0 CHECK
# ============================================================

#
# ACQ_I was all integer and so is ALQ_I.
#
# There are NO genuine decimal variables in this file.
#
# Therefore checking all 10 columns for unwanted ".0"
# is safe.
#
#
# Example:
#
#     SEQN:
#         83732.0 -> 83732
#
#     ALQ101:
#         1.0 -> 1
#
#     ALQ120Q:
#         0.0 -> 0
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "any ALQ_I variable."
)


# ============================================================
# 36. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 3,839 validated zero values."
)


# ============================================================
# 37. FINAL NONMISSING COUNTS
# ============================================================

EXPECTED_NONMISSING_COUNTS = {
    "ALQ101": 5208,
    "ALQ110": 1731,
    "ALQ120Q": 4224,
    "ALQ120U": 3376,
    "ALQ130": 3379,
    "ALQ141Q": 3377,
    "ALQ141U": 1340,
    "ALQ151": 4222,
    "ALQ160": 1352
}


print(
    "\n--- FINAL NONMISSING COUNTS ---"
)


for col, expected_count in EXPECTED_NONMISSING_COUNTS.items():

    values = raw_csv_df[col]


    actual_count = int(
        values
        .ne("")
        .sum()
    )


    print(
        f"{col}:",
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: final nonmissing count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Final nonmissing counts preserved."
)


# ============================================================
# 38. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 39. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 40. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "ALQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Alcohol Use"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Tiny artifact variables:",
    list(
        EXPECTED_TINY_COUNTS.keys()
    )
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "ALQ120Q restored zeros:",
    int(
        df["ALQ120Q"]
        .eq(0)
        .sum()
    )
)


print(
    "ALQ141Q restored zeros:",
    int(
        df["ALQ141Q"]
        .eq(0)
        .sum()
    )
)


print(
    "ALQ160 restored zeros:",
    int(
        df["ALQ160"]
        .eq(0)
        .sum()
    )
)


print(
    "Genuine decimal fields:",
    "NONE"
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "ALQ_I contains no genuine decimal variables."
)


print(
    "All 10 variables can safely be exported "
    "as whole-number IDs/codes after the "
    "validated tiny-zero repair."
)


print(
    "Your broader NHANES rule remains unchanged:"
)


print(
    "Genuine Dietary, Laboratory and Examination "
    "continuous decimal values remain exactly AS IS."
)

NHANES 2015-2016 ALQ_I XPT -> CSV
Rows: 5,735
Columns: 10
PASS: Structure = 5,735 rows x 10 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: ['SEQN', 'ALQ101', 'ALQ110', 'ALQ120Q', 'ALQ120U', 'ALQ130', 'ALQ141Q', 'ALQ141U', 'ALQ151', 'ALQ160']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 3,839
Variables containing tiny artifacts: 3

Tiny values by variable:
ALQ141Q    2025
ALQ160      969
ALQ120Q     845

PASS: Exact 3,839-value tiny-zero pattern across 3 variables validated.

--- VALIDATED ZERO RESTORATION ---
ALQ120Q: 845 restored to 0
ALQ141Q: 2,025 restored to 0
ALQ160: 969 restored to 0

PASS: 3,839 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
ALQ101: 0 fractional observations
ALQ110: 0 fractional observations
ALQ120Q: 0 fractional observations
ALQ120U: 0 fractio